In [74]:
!python --version

Python 3.12.13


In [75]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

In [76]:
df = pd.read_csv("Job_3_Resource_sentiment.csv")

In [77]:
df.head(5)

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [78]:
df.sample(10)

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
59414,3383,Facebook,Neutral,"Why Amazon, Facebook, and Google Simply Fell T..."
26750,996,AssassinsCreed,Positive,I won 4 achievements in Assassin's Creed Unity...
23784,4476,Google,Neutral,@ google sir please help me restore my Gmail a...
15367,3030,Dota2,Positive,soon I will finally be able to enjoy watching ...
32190,7524,LeagueOfLegends,Positive,I hereby condemn Hextech crystals because they...
47005,5665,HomeDepot,Negative,@ HomeDepot attention to executive administrat...
13151,8659,NBA2K,Positive,Greatest song of all time already IDC IDC IDC
8380,9437,Overwatch,Negative,Ah fuck Down we go again.
68937,3806,Cyberpunk2077,Positive,Oh Their chairs sure are 5 / 8 7 perfect score...
498,2486,Borderlands,Neutral,"Guns, Love, and Tentacles is out now, and here..."


In [79]:
df.shape

(74681, 4)

In [80]:
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   2401                                                   74681 non-null  int64 
 1   Borderlands                                            74681 non-null  object
 2   Positive                                               74681 non-null  object
 3   im getting on borderlands and i will murder you all ,  73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [82]:
df.describe()

,2401
count,74681.000000
mean,6432.640149
std,3740.423819
min,1.000000
25%,3195.000000
50%,6422.000000
75%,9601.000000
max,13200.000000


In [83]:
df.rename(columns={'Positive': 'sentiment'}, inplace=True)

In [84]:
df

,2401,Borderlands,sentiment,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [85]:
df.rename(columns={'im getting on borderlands and i will murder you all ,': 'text'}, inplace=True)

In [86]:
df.sample(5)

,2401,Borderlands,sentiment,text
60655,3593,Facebook,Negative,It was useless... Who is the Facebook manager?
54992,2243,CallOfDuty,Irrelevant,Sound ON! Enjoy the groove and this little pla...
995,2577,Borderlands,Positive,Who's down for some @Borderlands on
30256,800,ApexLegends,Positive,NaN
18572,9983,PlayStation5(PS5),Neutral,The size difference between a Xbox and a Ps5 w...


In [87]:
df = df[['text', 'sentiment']]

In [88]:
df

,text,sentiment
0,I am coming to the borders and I will kill you...,Positive
1,im getting on borderlands and i will kill you ...,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive
...,...,...
74676,Just realized that the Windows partition of my...,Positive
74677,Just realized that my Mac window partition is ...,Positive
74678,Just realized the windows partition of my Mac ...,Positive
74679,Just realized between the windows partition of...,Positive


In [89]:
df.shape

(74681, 2)

In [90]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       73995 non-null  object
 1   sentiment  74681 non-null  object
dtypes: object(2)
memory usage: 1.1+ MB


In [91]:
df.describe()

,text,sentiment
count,73995,74681
unique,69490,4
top,,Negative
freq,172,22542


In [92]:
df.columns

Index(['text', 'sentiment'], dtype='object')

In [93]:
df.isnull().sum()

,0
text,686
sentiment,0


In [94]:
print(df['sentiment'].value_counts())

sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [95]:
df.duplicated().sum()

np.int64(4909)

In [96]:
df.drop_duplicates(inplace=True)

/tmp/ipykernel_2837/3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [97]:
df.duplicated().sum()

np.int64(0)

In [98]:
df.dropna(inplace=True)

/tmp/ipykernel_2837/1379821321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [99]:
df.isnull().sum()

,0
text,0
sentiment,0


In [100]:
df.shape

(69768, 2)

In [101]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69768 entries, 0 to 74680
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       69768 non-null  object
 1   sentiment  69768 non-null  object
dtypes: object(2)
memory usage: 1.6+ MB


In [102]:
df.describe()

,text,sentiment
count,69768,69768
unique,69490,4
top,by,Negative
freq,4,21237


In [103]:
df.sample(10)

,text,sentiment
44903,maaaan fuck verizon a bitch is abt to go tf of...,Negative
5820,Across the Fourwinds is the first book in the ...,Neutral
17432,okay and we can have shit,Negative
73116,... Why Nvidia's GeForce Now is so controversi...,Neutral
27138,° _ ° * Gets bad flashback in the cinema *.:) ...,Irrelevant
21292,he,Irrelevant
32097,Thank DRX a bit bit for careless in these last...,Neutral
51893,people really said screw sadie adler,Neutral
38952,All these videos really excite me about LOR. I...,Positive
37543,Liked on YouTube: Hearthstone Felfire Festival...,Irrelevant


In [104]:
df['text'] = df['text'].astype(str)

/tmp/ipykernel_2837/1180961456.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].astype(str)


In [105]:
df['text']

,text
0,I am coming to the borders and I will kill you...
1,im getting on borderlands and i will kill you ...
2,im coming on borderlands and i will murder you...
3,im getting on borderlands 2 and i will murder ...
4,im getting into borderlands and i can murder y...
...,...
74676,Just realized that the Windows partition of my...
74677,Just realized that my Mac window partition is ...
74678,Just realized the windows partition of my Mac ...
74679,Just realized between the windows partition of...


In [106]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

/tmp/ipykernel_2837/1069262265.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].apply(clean_text)


In [107]:
df['text']

,text
0,i am coming to the borders and i will kill you...
1,im getting on borderlands and i will kill you all
2,im coming on borderlands and i will murder you...
3,im getting on borderlands and i will murder y...
4,im getting into borderlands and i can murder y...
...,...
74676,just realized that the windows partition of my...
74677,just realized that my mac window partition is ...
74678,just realized the windows partition of my mac ...
74679,just realized between the windows partition of...


In [108]:
encoder = LabelEncoder()
df['sentiment_encoded'] = encoder.fit_transform(df['sentiment'])

print("Label mapping:")
for i, c in enumerate(encoder.classes_):
    print(i, "->", c)

Label mapping:
0 -> Irrelevant
1 -> Negative
2 -> Neutral
3 -> Positive


/tmp/ipykernel_2837/3100372417.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment_encoded'] = encoder.fit_transform(df['sentiment'])


In [109]:
encoder

LabelEncoder()

In [110]:
encoder.classes_

array(['Irrelevant', 'Negative', 'Neutral', 'Positive'], dtype=object)

In [111]:
df

,text,sentiment,sentiment_encoded
0,i am coming to the borders and i will kill you...,Positive,3
1,im getting on borderlands and i will kill you all,Positive,3
2,im coming on borderlands and i will murder you...,Positive,3
3,im getting on borderlands and i will murder y...,Positive,3
4,im getting into borderlands and i can murder y...,Positive,3
...,...,...,...
74676,just realized that the windows partition of my...,Positive,3
74677,just realized that my mac window partition is ...,Positive,3
74678,just realized the windows partition of my mac ...,Positive,3
74679,just realized between the windows partition of...,Positive,3


In [112]:
max_words = 20000
max_length = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="")
tokenizer.fit_on_texts(df['text'])

X = pad_sequences(
    tokenizer.texts_to_sequences(df['text']),
    maxlen=max_length,
    padding='post'
)

y = df['sentiment_encoded'].values

In [113]:
tokenizer

In [114]:
X

array([[   3,   98,  370, ...,    0,    0,    0],
       [  31,  155,   14, ...,    0,    0,    0],
       [  31,  370,   14, ...,    0,    0,    0],
       ...,
       [  21, 1830,    2, ...,    0,    0,    0],
       [  21, 1830,  683, ...,    0,    0,    0],
       [  21,   32,    2, ...,    0,    0,    0]], dtype=int32)

In [115]:
X.shape

(69768, 100)

In [116]:
y

array([3, 3, 3, ..., 3, 3, 3])

In [117]:
y.shape

(69768,)

In [118]:
df

,text,sentiment,sentiment_encoded
0,i am coming to the borders and i will kill you...,Positive,3
1,im getting on borderlands and i will kill you all,Positive,3
2,im coming on borderlands and i will murder you...,Positive,3
3,im getting on borderlands and i will murder y...,Positive,3
4,im getting into borderlands and i can murder y...,Positive,3
...,...,...,...
74676,just realized that the windows partition of my...,Positive,3
74677,just realized that my mac window partition is ...,Positive,3
74678,just realized the windows partition of my mac ...,Positive,3
74679,just realized between the windows partition of...,Positive,3


In [119]:
X.dtype

dtype('int32')

In [120]:
y.dtype

dtype('int64')

In [121]:
X.ndim

2

In [122]:
y.ndim

1

In [123]:
X.nbytes

27907200

In [124]:
y.nbytes

558144

In [125]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("\nTrain:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Train: (48837, 100)
Validation: (10465, 100)
Test: (10466, 100)


In [126]:
X_train

array([[    1,   211,    27, ...,     0,     0,     0],
       [    3,   799,   578, ...,     0,     0,     0],
       [  436,  1394,  5377, ...,     0,     0,     0],
       ...,
       [  151,    36,   198, ...,     0,     0,     0],
       [  394,   548,   370, ...,     0,     0,     0],
       [  136,   196, 13157, ...,     0,     0,     0]], dtype=int32)

In [127]:
X_temp

array([[1463,  210,   53, ...,    0,    0,    0],
       [ 537,   40,  129, ...,    0,    0,    0],
       [  15,  126,  651, ...,    0,    0,    0],
       ...,
       [   7,   83,   86, ...,    0,    0,    0],
       [   3,   98,    3, ...,    0,    0,    0],
       [ 589,    3,   21, ...,    0,    0,    0]], dtype=int32)

In [128]:
X_test

array([[  35,   54,  126, ...,    0,    0,    0],
       [  11,    8,   29, ...,    0,    0,    0],
       [1994,  918, 7480, ...,    0,    0,    0],
       ...,
       [ 151,   60,   58, ...,    0,    0,    0],
       [7992,   72, 1293, ...,    0,    0,    0],
       [ 857, 2648,  211, ...,    0,    0,    0]], dtype=int32)

In [129]:
X_val

array([[   2, 1891,  228, ...,    0,    0,    0],
       [   5,    3,   32, ...,    0,    0,    0],
       [  90, 7372,  744, ...,    0,    0,    0],
       ...,
       [  19,   15, 2471, ...,    0,    0,    0],
       [   3,  226, 2177, ...,    0,    0,    0],
       [   3,  192, 1119, ...,    0,    0,    0]], dtype=int32)

In [130]:
y_temp

array([3, 1, 3, ..., 1, 3, 0])

In [131]:
y_test

array([3, 3, 2, ..., 1, 2, 1])

In [132]:
y_train

array([1, 0, 0, ..., 3, 1, 3])

In [133]:
y_val

array([3, 2, 1, ..., 2, 3, 0])

In [134]:
y_train.shape

(48837,)

In [135]:
print(np.bincount(y_train))

[ 8598 14866 11977 13396]


In [136]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))
print("\nClass Weights:", class_weights_dict)



Class Weights: {0: np.float64(1.4200104675505931), 1: np.float64(0.821286829005785), 2: np.float64(1.0193913333889957), 3: np.float64(0.9114101224246044)}


In [137]:
class_weights

array([1.42001047, 0.82128683, 1.01939133, 0.91141012])

In [138]:
print(np.bincount(y_train))

[ 8598 14866 11977 13396]
